<a href="https://colab.research.google.com/github/gbcancian/data_analysis_treinos/blob/main/Academia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Quantidade de treinos por semana
# 2. Exercícios mais utilizados
# 3. Quantidade média de repetições por exercício
# 4. Evolução de carga (Progressive overload)
# 5. Volume total de treino
# 6. Volume por treino
# 7. Tempo médio de treino


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

treinos = pd.read_csv('workout_data.csv')

In [ ]:
#Renomeando as colunas do DataFrame
treinos.columns=['titulo','data_inicio','data_final','descricao','exercicio','superset_id','notas_exercicio','indice_serie','tipo_serie','peso','repeticoes','distancia','duracao','rpe']
treinos.head()

,titulo,data_inicio,data_final,descricao,exercicio,superset_id,notas_exercicio,indice_serie,tipo_serie,peso,repeticoes,distancia,duracao,rpe
0,Seg - Push,"9 Mar 2026, 17:05","9 Mar 2026, 18:31",NaN,Desenvolvimento Arnold (halteres),NaN,NaN,0,normal,14.0,12.0,NaN,0.0,NaN
1,Seg - Push,"9 Mar 2026, 17:05","9 Mar 2026, 18:31",NaN,Desenvolvimento Arnold (halteres),NaN,NaN,1,normal,20.0,12.0,NaN,0.0,NaN
2,Seg - Push,"9 Mar 2026, 17:05","9 Mar 2026, 18:31",NaN,Desenvolvimento Arnold (halteres),NaN,NaN,2,normal,24.0,10.0,NaN,0.0,NaN
3,Seg - Push,"9 Mar 2026, 17:05","9 Mar 2026, 18:31",NaN,Desenvolvimento Arnold (halteres),NaN,NaN,3,normal,26.0,8.0,NaN,0.0,NaN
4,Seg - Push,"9 Mar 2026, 17:05","9 Mar 2026, 18:31",NaN,Elevação Lateral (Halter),NaN,NaN,0,normal,9.0,12.0,NaN,0.0,NaN


In [ ]:
#Verificando se possui pesos nulos
treinos.query('peso == 0 | peso < 0 | peso.isnull()')

,titulo,data_inicio,data_final,exercicio,indice_serie,tipo_serie,peso,repeticoes,duracao
15,Seg - Push,"9 Mar 2026, 17:05","9 Mar 2026, 18:31",Pec Deck,0,normal,NaN,0.0,0.0
16,Seg - Push,"9 Mar 2026, 17:05","9 Mar 2026, 18:31",Pec Deck,1,normal,NaN,0.0,0.0
17,Seg - Push,"9 Mar 2026, 17:05","9 Mar 2026, 18:31",Pec Deck,2,normal,NaN,0.0,0.0
18,Seg - Push,"9 Mar 2026, 17:05","9 Mar 2026, 18:31",Leg Press 45º (Máquina),0,normal,NaN,0.0,0.0
19,Seg - Push,"9 Mar 2026, 17:05","9 Mar 2026, 18:31",Leg Press 45º (Máquina),1,normal,NaN,0.0,0.0
20,Seg - Push,"9 Mar 2026, 17:05","9 Mar 2026, 18:31",Leg Press 45º (Máquina),2,normal,NaN,0.0,0.0
39,Seg - Push,"2 Mar 2026, 16:11","2 Mar 2026, 17:47",Pec Deck,0,normal,NaN,0.0,0.0
40,Seg - Push,"2 Mar 2026, 16:11","2 Mar 2026, 17:47",Pec Deck,1,normal,NaN,0.0,0.0
41,Seg - Push,"2 Mar 2026, 16:11","2 Mar 2026, 17:47",Pec Deck,2,normal,NaN,0.0,0.0
114,"Qua - Lower A (Quadriceps, Panturrilha, Gluteo)","4 Jun 2025, 17:12","4 Jun 2025, 18:23",Peso Morto Com Pernas Esticadas,0,normal,NaN,12.0,NaN


In [ ]:
#Visualizando quais colunas não tem dados
treinos.query('descricao.notnull()')
treinos.query('superset_id.notnull()')
treinos.query('notas_exercicio.notnull()')
treinos.query('distancia.notnull()')
treinos.query('duracao != 0 & duracao.notnull()') #manter duração
treinos.query('rpe != 0 & rpe.notnull()')

,titulo,data_inicio,data_final,descricao,exercicio,superset_id,notas_exercicio,indice_serie,tipo_serie,peso,repeticoes,distancia,duracao,rpe


In [ ]:
#Criando uma variavel com as colunas utilizaveis
treinos = treinos[['titulo','data_inicio','data_final','exercicio','indice_serie','tipo_serie','peso','repeticoes','duracao']]

In [ ]:
#DataFrame tratado
treinos_validos = treinos.query('peso > 0')
treinos_validos.head()
treinos_validos.info()

<class 'pandas.core.frame.DataFrame'>
Index: 290 entries, 0 to 303
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   titulo        290 non-null    object 
 1   data_inicio   290 non-null    object 
 2   data_final    290 non-null    object 
 3   exercicio     290 non-null    object 
 4   indice_serie  290 non-null    int64  
 5   tipo_serie    290 non-null    object 
 6   peso          290 non-null    float64
 7   repeticoes    290 non-null    float64
 8   duracao       151 non-null    float64
dtypes: float64(3), int64(1), object(5)
memory usage: 22.7+ KB


In [ ]:
#Formantando datas para o formato padrão do pandas
treinos_validos['data_inicio'] = pd.to_datetime(treinos_validos['data_inicio'], format='%d %b %Y, %H:%M', dayfirst=True, errors='coerce')
treinos_validos['data_final'] = pd.to_datetime(treinos_validos['data_final'], format='%d %b %Y, %H:%M', dayfirst=True, errors='coerce')
treinos_por_dia = treinos_validos.groupby('data_inicio')
treinos_por_dia.head()

,titulo,data_inicio,data_final,exercicio,indice_serie,tipo_serie,peso,repeticoes,duracao
0,Seg - Push,2026-03-09 17:05:00,2026-03-09 18:31:00,Desenvolvimento Arnold (halteres),0,normal,14.0,12.0,0.0
1,Seg - Push,2026-03-09 17:05:00,2026-03-09 18:31:00,Desenvolvimento Arnold (halteres),1,normal,20.0,12.0,0.0
2,Seg - Push,2026-03-09 17:05:00,2026-03-09 18:31:00,Desenvolvimento Arnold (halteres),2,normal,24.0,10.0,0.0
3,Seg - Push,2026-03-09 17:05:00,2026-03-09 18:31:00,Desenvolvimento Arnold (halteres),3,normal,26.0,8.0,0.0
4,Seg - Push,2026-03-09 17:05:00,2026-03-09 18:31:00,Elevação Lateral (Halter),0,normal,9.0,12.0,0.0
24,Seg - Push,2026-03-02 16:11:00,2026-03-02 17:47:00,Desenvolvimento Arnold (halteres),0,normal,14.0,12.0,0.0
25,Seg - Push,2026-03-02 16:11:00,2026-03-02 17:47:00,Desenvolvimento Arnold (halteres),1,normal,20.0,10.0,0.0
26,Seg - Push,2026-03-02 16:11:00,2026-03-02 17:47:00,Desenvolvimento Arnold (halteres),2,normal,22.0,8.0,0.0
27,Seg - Push,2026-03-02 16:11:00,2026-03-02 17:47:00,Desenvolvimento Arnold (halteres),3,normal,24.0,8.0,0.0
28,Seg - Push,2026-03-02 16:11:00,2026-03-02 17:47:00,Elevação Lateral (Halter),0,normal,8.0,15.0,0.0
